In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")


CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
yaml_path = "/home/samuel/Downloads/download/full_dataset_new_version/dataset_new_version/combined_dataset_normalized/sorted/dataset.yaml"

In [4]:

#!/usr/bin/env python3

from ultralytics import YOLO

model = YOLO("yolo12n.pt")  # works out-of-the-box

results = model.train(
    data=yaml_path,
    epochs=30,        # adjust depending on dataset size
    imgsz=1024,        # big enough for small plants
    batch=4,         # adjust based on GPU memory
    device=0,
    patience=30,       # early stopping
    freeze=0,          # fine-tune everything
    mosaic=0,          # turn off mosaic
    mixup=0,           # turn off mixup
    hsv_h=0, hsv_s=0, hsv_v=0,   # no color augmentation
    degrees=0, scale=0, translate=0, shear=0, perspective=0,
    val=True   # <-- ensures validation is computed
)

print("Training complete.")

Ultralytics 8.4.12 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5781MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/samuel/Downloads/download/full_dataset_new_version/dataset_new_version/combined_dataset_normalized/sorted/dataset.yaml, degrees=0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=0, multi_scale=0.0, name=train, nbs=64, 

In [11]:
# test on the test set
import os
from pathlib import Path
from unittest import result
from ultralytics import YOLO

test_image = "/home/samuel/Downloads/download/full_dataset_new_version/dataset_new_version/combined_dataset_normalized/sorted/images/test/"
images = os.listdir(test_image)
num_images_to_check = 10
print(images[:num_images_to_check])  # print first 10 image names

model = YOLO("/home/samuel/MasterThesis/runs/detect/train_RED_REDEDGE_NIR/weights/best.pt")
run: int = 0
prediction_output = Path(test_image).parent.parent / "predictions"

if prediction_output.exists():
    run = len(list(prediction_output.glob("run*")))

prediction_output = prediction_output / f"run{run}"
while prediction_output.exists():
    run += 1
    prediction_output = prediction_output.parent / f"run{run}"
prediction_output.mkdir(parents=True, exist_ok=True)

for img_name in images:
    img_path = os.path.join(test_image, img_name)
    results = model.predict(source=img_path, conf=0.25, save=True)

    prediction_file = img_name.replace(".png", ".txt")

    for result in results:
        if not result.boxes:
            continue

        for box in result.boxes:
            cls = int(box.cls[0])
            conf = box.conf[0]
            xyxy = box.xyxy[0].tolist()
            with open (prediction_output / prediction_file, "a") as f:
                f.write(f"{cls} {conf:.4f} {' '.join(f'{coord:.2f}' for coord in xyxy)}\n")

    print(f"Inference complete for {img_name}. Results saved.")


# results = model.predict(source=test_image, conf=0.25, save=True)
print("Inference complete. Results saved.")

['Bjornkjaervej_TestFlight_2_mid_tile_9_8.png', 'Bjornkjaervej_TestFlight_2_mid_tile_22_3.png', 'Bjornkjaervej_TestFlight_2_bigger_tile_9_9.png', 'Bjornkjaervej_TestFlight_2_bigger_tile_7_12.png', 'Bjornkjaervej_TestFlight_2_bigger_tile_13_12.png', 'Bjornkjaervej_TestFlight_2_small_tile_16_11.png', 'Bjornkjaervej_TestFlight_2_bigger_tile_14_6.png', 'Bjornkjaervej_TestFlight_2_small_tile_5_5.png', 'Bjornkjaervej_TestFlight_2_bigger_tile_21_4.png', 'Bjornkjaervej_TestFlight_2_bigger_tile_20_4.png']

image 1/1 /home/samuel/Downloads/download/full_dataset_new_version/dataset_new_version/combined_dataset_normalized/sorted/images/test/Bjornkjaervej_TestFlight_2_mid_tile_9_8.png: 1024x1024 2 Potatoess, 13.4ms
Speed: 2.1ms preprocess, 13.4ms inference, 0.9ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /home/samuel/MasterThesis/runs/detect/predict
Inference complete for Bjornkjaervej_TestFlight_2_mid_tile_9_8.png. Results saved.

image 1/1 /home/samuel/Downloads/download/

In [2]:
from ultralytics import YOLO
import cv2
import numpy as np
from pathlib import Path

# Load your best checkpoint
model = YOLO("/home/samuel/MasterThesis/runs/obb/train2/weights/best.pt")

# Run inference on training set
train_results = model.predict(
    source="/home/samuel/test/MasterThesis/Orthomosaics/large/original/original/combined_output/NIR_RED_NGRDI",
    imgsz=1024,
    conf=0.25,  # Confidence threshold - adjust based on what you see
    iou=0.7,
    save=True,  # Saves visualizations
    save_txt=True,  # Saves prediction labels
    save_conf=True,  # Includes confidence scores
    project="/home/samuel/MasterThesis/runs/obb/inference",
    name="train_predictions",
    exist_ok=True
)

# Run inference on validation set
val_results = model.predict(
    source="/home/samuel/test/MasterThesis/Orthomosaics/dataset/images/val",
    imgsz=1024,
    conf=0.25,
    iou=0.7,
    save=True,
    save_txt=True,
    save_conf=True,
    project="/home/samuel/MasterThesis/runs/obb/inference",
    name="val_predictions",
    exist_ok=True
)

# Run inference on test set
test_results = model.predict(
    source="/home/samuel/test/MasterThesis/Orthomosaics/dataset/images/test",
    imgsz=1024,
    conf=0.25,
    iou=0.7,
    save=True,
    save_txt=True,
    save_conf=True,
    project="/home/samuel/MasterThesis/runs/obb/inference",
    name="test_predictions",
    exist_ok=True
)

print("Inference complete!")
print(f"Train predictions saved to: runs/obb/inference/train_predictions")
print(f"Val predictions saved to: runs/obb/inference/val_predictions")
print(f"Test predictions saved to: runs/obb/inference/test_predictions")


image 1/469 /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/combined_output/NIR_RED_NGRDI/Bjornkjaervej_TestFlight_2_bigger_tile_10_10_NIR_RED_NGRDI.png: 1024x1024 11 Potatoess, 8.5ms
image 2/469 /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/combined_output/NIR_RED_NGRDI/Bjornkjaervej_TestFlight_2_bigger_tile_10_11_NIR_RED_NGRDI.png: 1024x1024 7 Potatoess, 9.6ms
image 3/469 /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/combined_output/NIR_RED_NGRDI/Bjornkjaervej_TestFlight_2_bigger_tile_10_12_NIR_RED_NGRDI.png: 1024x1024 2 Potatoess, 8.1ms
image 4/469 /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/combined_output/NIR_RED_NGRDI/Bjornkjaervej_TestFlight_2_bigger_tile_10_13_NIR_RED_NGRDI.png: 1024x1024 3 Potatoess, 7.9ms
image 5/469 /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/combined_output/NIR_RED_NGRDI/Bjornkjaervej_TestFlight_2_bigger_tile_10_2_NIR_RED_NGRDI.png: 1024x1024 3 P

In [13]:
from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")
# Display model information (optional)
model.info()

# Train the model on the COCO8 example dataset for 100 epochs
results = model.train(
    data=yaml_path,
    epochs=100,        # adjust depending on dataset size
    imgsz=1024,        # big enough for small plants
    batch=1,
    device=0,
    patience=30,       # early stopping
    val=True   # <-- ensures validation is computed
    )

# Run inference with the RT-DETR-l model on the 'bus.jpg' image
results = model("/home/samuel/MasterThesis/AI")

rt-detr-l summary: 449 layers, 32,970,476 parameters, 0 gradients, 108.3 GFLOPs
New https://pypi.org/project/ultralytics/8.4.0 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5781MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/samuel/Downloads/dataset_new_version/combined_dataset_normalized/sorted/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=

/home/samuel/MasterThesis/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      4.38G     0.7129       1.75     0.1867          0       1024: 100% ━━━━━━━━━━━━ 616/616 4.6it/s 2:15<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 110/110 11.6it/s 9.5s0.2s
                   all        220        723      0.611       0.52       0.57      0.338

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      2.73G     0.5256      1.427    0.03599         12       1024: 0% ──────────── 0/616  0.2s

/home/samuel/MasterThesis/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      2.74G     0.4269     0.6758    0.05897          7       1024: 100% ━━━━━━━━━━━━ 616/616 4.6it/s 2:14<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 110/110 11.4it/s 9.7s0.2s
                   all        220        723      0.567       0.53      0.525      0.315

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      2.83G     0.3213     0.8185    0.02939          3       1024: 0% ──────────── 0/616  0.2s

...

/home/samuel/MasterThesis/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     99/100      2.81G     0.1888     0.2981    0.02047          1       1024: 100% ━━━━━━━━━━━━ 616/616 5.1it/s 2:02<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 110/110 12.0it/s 9.1s0.1s
                   all        220        723      0.798      0.808      0.859      0.621

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    100/100      2.98G      1.123     0.3555     0.1658          1       1024: 0% ──────────── 0/616  0.2s

/home/samuel/MasterThesis/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:148.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    100/100         3G     0.1844     0.2882    0.02236          5       1024: 100% ━━━━━━━━━━━━ 616/616 5.0it/s 2:02<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 110/110 12.1it/s 9.1s0.1s
                   all        220        723      0.783      0.797      0.846      0.617

100 epochs completed in 3.868 hours.
Optimizer stripped from /home/samuel/MasterThesis/runs/detect/train6/weights/last.pt, 66.3MB
Optimizer stripped from /home/samuel/MasterThesis/runs/detect/train6/weights/best.pt, 66.3MB

Validating /home/samuel/MasterThesis/runs/detect/train6/weights/best.pt...
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5781MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 110/110 13.9it/s 7.9s0.1s
                   all        220

FileNotFoundError: No images or videos found in /home/samuel/MasterThesis/AI. Supported formats are:
images: {'pfm', 'dng', 'tif', 'png', 'jpg', 'bmp', 'webp', 'jpeg', 'tiff', 'heic', 'mpo'}
videos: {'ts', 'mov', 'wmv', 'avi', 'mkv', 'mpg', 'webm', 'asf', 'mp4', 'mpeg', 'm4v', 'gif'}